## Prepare python environment


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
random_state=5 # use this to control randomness across runs e.g., dataset partitioning

## Preparing the Glass Dataset (2 points)

---


We will use glass dataset from UCI machine learning repository. Details for this data can be found [here](https://archive.ics.uci.edu/dataset/42/glass+identification).
The objective of the dataset is to identify the class of glass based on the following features:

1. RI: refractive index
2. Na: Sodium
3. Mg: Magnesium
4. Al: Aluminum
5. Si: Silica
6. K: Potassium
7. Ca: Calcium
8. Ba: Barium
9. Fe: Iron
10. Type of glass (Target label)

The classes of glass are:

1. building_windows_float_processed
2. building_windows_non_float_processed
3. vehicle_windows_float_processed
4. containers
5. tableware
6. headlamps

Identification of glass from its content can be used for forensic analysis.

### Loading the dataset

In [ ]:
# Download and load the dataset
import os
if not os.path.exists('glass.csv'):
    !wget https://raw.githubusercontent.com/JHA-Lab/ece364_2026/master/data/glass.csv
data = pd.read_csv('glass.csv')

# Display the first five instances in the dataset
data.head(5)

# Adding Additional Features

1. Ca_Na: Calcium Sodium
2. Al_Mg: Aluminum Magnesium
3. Ca_Mg: Calcium Magnesium

In [ ]:
# Additional features to be added to the data
data['Ca_Na'] = data.Ca*data.Na
data['Al_Mg'] = data.Al*data.Mg
data['Ca_Mg'] = data.Ca*data.Mg

# get column names
col_names = data.columns

### Check the data type for each column

In [ ]:
data.info()

#### Look at some statistics of the data using the `describe` function in pandas.

In [ ]:
data.describe()

1. Count tells us the number of Non-empty rows in a feature.

2. Mean tells us the mean value of that feature.

3. Std tells us the Standard Deviation Value of that feature.

4. Min tells us the minimum value of that feature.

5. 25%, 50%, and 75% are the percentile/quartile of each feature.

6. Max tells us the maximum value of that feature.

### Visualize the Data

#### Check how many instances of each target class are there in the data. This has been done for you.

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.8)
plt.subplots(figsize = (15,8))
sns.countplot(x='Type',data=data).set_title('Count of Glass Types')
plt.xlabel('Target (Classes of Glass)')
plt.ylabel('Count')

#### Calculate `mean` values for each feature grouped by target. This has been done for you.

In [ ]:
# Compute mean values for each feature grouped by target
data.groupby('Type', as_index=False).mean()

#### Create box plots to see distribution of each feature. See [here](https://seaborn.pydata.org/generated/seaborn.boxplot.html) for further details. This has been done for you.

In [ ]:
sns.set_theme(style="white", font_scale=1.2)
plt.subplots(figsize = (30,20))
plt.subplot(3,4,1)
sns.boxplot(x='Type', y='RI', data=data)
plt.subplot(3,4,2)
sns.boxplot(x='Type', y='Na', data=data)
plt.subplot(3,4,3)
sns.boxplot(x='Type', y='Mg', data=data)
plt.subplot(3,4,4)
sns.boxplot(x='Type', y='Al', data=data)
plt.subplot(3,4,5)
sns.boxplot(x='Type', y='Si', data=data)
plt.subplot(3,4,6)
sns.boxplot(x='Type', y='K', data=data)
plt.subplot(3,4,7)
sns.boxplot(x='Type', y='Ca', data=data)
plt.subplot(3,4,8)
sns.boxplot(x='Type', y='Ba', data=data)
plt.subplot(3,4,9)
sns.boxplot(x='Type', y='Fe', data=data)
plt.subplot(3,4,10)
sns.boxplot(x='Type', y='Ca_Na', data=data)
plt.subplot(3,4,11)
sns.boxplot(x='Type', y='Al_Mg', data=data)
plt.subplot(3,4,12)
sns.boxplot(x='Type', y='Ca_Mg', data=data)
plt.show()

#### Create a pairplot to display pairwise relationship. See [here](https://seaborn.pydata.org/generated/seaborn.pairplot.html) for further details. This has been done for you.

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
# Select a subset of features for visualization
sns.pairplot(data[['RI','Na','Mg','Al','Si','Type']], hue='Type')

In [ ]:
# Plot heatmap showing correlation between different features
plt.subplots(figsize=(15,10))
sns.heatmap(data.corr(),cmap='YlGnBu',annot=True, linewidth=.5)

### Extract target and descriptive features (1 point)

#### Separate the target and features from the data.

In [ ]:
# Store all the features from the data in X
X = # TODO

# Store all the labels in y
y = # TODO

In [ ]:
# Convert data to numpy array
X = # TODO
y = # TODO

### Create training and validation datasets (1 point)


We will split the dataset into training and validation set. Generally in machine learning, we split the data into training,
validation and test set (this will be covered in later chapters). The model with best performance on the validation set is used to evaluate perfromance on
the test set which is the unseen data. In this assignment, we will using `train set` for training and evaluate the performance on the `validation set` for various
model configurations to determine the best hyperparameters (parameter setting yielding the best performance).

Split the data into training and validation set using `train_test_split`.  See [here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) for details. To get consistent result while splitting, set `random_state` to the value defined earlier. We use 80% of the data for training and 20% of the data for validation. This has been done for you.

In [ ]:
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=.2,random_state=random_state) # 80% training and 20% validation

## Training Decision Tree-based Classifiers (18 points)


### Exercise 1: Learning a Decision Tree (10 points)

#### We will use the `sklearn` library to train a Decision Tree classifier. Review ch.4 and see [here](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) for more details.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# tree visualization helper function
from sklearn.tree import export_graphviz
from six import StringIO
from IPython.display import Image
import pydotplus

"""
clf: DecisionTreeClassifier

Returns a bytes object representing the image of the tree
"""
def get_tree_image(clf):
    dot_data = StringIO()
    feature_names=data.drop('Type',axis=1).columns
    class_names=["Building_windows_float_processed", "Building_windows_non_float_processed", "Vehicle_windows_float_processed", "Containers", "Tableware", "Headlamps"]
    export_graphviz(clf, out_file=dot_data,
                    filled=True, rounded=True,
                    special_characters=True,
                    feature_names=feature_names,
                    class_names=class_names)
    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())


    return graph.create_png()

#### Exercise 1a: Fit and interpret a decision tree. (6 points)

#### Fit Decision trees using the Gini index and entropy-based impurity measure.

#### Set the random_state to the value defined above. Keep all other parameters at their default values.

#### Report the training and validation set accuracies for each classifier.

In [ ]:
# Create decision tree classifiers using both criteria

#### Visualize the Decision Tree with the best validation performance.

In [ ]:
best_clf=entropy_clf
tree_image=get_tree_image(best_clf)
Image(tree_image)

In [ ]:
best_clf=gini_clf
tree_image=get_tree_image(best_clf)
Image(tree_image)

#### Indicate the most informative descriptive feature (with the threshold) and briefly explain why this is the most informative (from an algorithmic viewpoint).

**Answer:** [TODO]

#### Briefly comment on the tree's depth and what factors may contribute to the shallowness/complexity of the tree.


**Answer:** [TODO]



#### Show how one can interpret the tree by specifying the rule from its left most branch.

**Answer:** [TODO]

#### Exercise 1b: Preprune a decision tree. (2 points)

#### Next, let's try pruning the tree to see if we can improve the classifier's generalization performance.

#### Preprune a decision tree by varying the `max_depth` among {None (no depth control), 1,3,5,7}.

#### Set the criterion to entropy and the random_state to the value defined above. Keep all other parameters at their default values.

#### Report the training and validation set accuracies for each classifier.

In [ ]:
# TODO

#### Analyze the effect of increasing tree depth on training and validation performance.

**Answer:** [TODO]

#### Exercise 1c: Postprune a decision tree. (2 points)

#### Implement reduced error pruning on the two decision trees that you previously trained in Exercise 1a. Use the validation set to decide which nodes to prune.

#### Report the validation set accuracies for each classifier after pruning.

#### The pruning function is partly writen, please fill in the TODO part. Review ch.4.4.4 for more details.


In [ ]:
# Fit new trees if you want to keep the trees in 1a unmodified
gini_clf = # TODO: your tree fitted using the Gini index
entropy_clf = # TODO: your tree fitted using the entropy-based impurity measures

def reduced_error_pruning(dtree, X_val, y_val):
    """
    Perform reduced error pruning on a decision tree classifier.

    This function directly modifies the decision tree passed as an argument.

    Args:
    dtree: DecisionTreeClassifier
        The decision tree to prune, must be already fitted.
    X_val: array-like
        Validation features used to prune the tree.
    y_val: array-like
        Validation labels used to determine pruning effectiveness.

    Returns:
    dtree: DecisionTreeClassifier
        The pruned decision tree.
    """

    # Access the internal tree structure to identify non-leaf nodes
    non_leaf_nodes = [i for i in range(dtree.tree_.node_count) if dtree.tree_.children_left[i] != dtree.tree_.children_right[i]]

    # Track the best accuracy and corresponding tree configuration. initialize with original accuracy.
    best_acc = # TODO

    # Iterate over non-leaf nodes in reverse order to consider pruning from the bottom up
    for i in reversed(non_leaf_nodes):
        # Store current node children to restore if needed
        left, right = dtree.tree_.children_left[i], dtree.tree_.children_right[i]

        # Temporarily make the node a leaf
        dtree.tree_.children_left[i], dtree.tree_.children_right[i] = -1, -1

        # Calculate the accuracy of the tree with the node pruned (turned into a leaf)
        temp_acc = # TODO

        if temp_acc < best_acc: # Revert pruning if accuracy decreases
            # Restore the node to its original state
            # TODO
        else:
            # Update the best accuracy observed
            # TODO

    return dtree  # Return the modified tree


pruned_entropy = reduced_error_pruning(entropy_clf, X_val, y_val)
pruned_entropy_acc = # TODO
print(f"Validation accuracy of entropy tree after pruning: {pruned_entropy_acc:.4f}")

pruned_gini = reduced_error_pruning(gini_clf, X_val, y_val)
pruned_gini_acc = # TODO
print(f"Validation accuracy of gini tree after pruning: {pruned_gini_acc:.4f}")

### Exercise 2: Learning an Ensemble of Decision Trees (8 points)

#### We will use the `sklearn` library to implement bagging and boosting. Review ch.4 and read more on [bagging](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) and [boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

#### Exercise 2a: Fit a Random Forest. (4 points)

#### Fit different Random Forest classifiers by varying the number of trees among {10, 50, 100, 400, 1000}.

#### Set the `criterion` to entropy and set the random_state to the value defined above. Keep all other parameters at their default values.

#### Report the validation set accuracies for each classifier.

In [ ]:
# TODO

#### Comment on the effect of increasing the number of trees on validation performance. Compare the performance of the best performing Random Forest classifier against the Decision Tree Classifier trained with entropy (Ex. 1a) and explain any difference.

**Answer:** [TODO]

#### Exercise 2b: Fit a Gradient Boosted Decision Tree (GBDT). (4 points)

#### Fit different GBDTs by varying the number of boosting steps/trees added among {5, 10, 20, 50, 100, 200}.

#### Set the `n_iter_no_change` to 100, `validation_fraction=0.2`, and random_state to the value defined above. Keep all other parameters at their default values.

#### Report the training and validation set accuracies for each classifier.

In [ ]:
# TODO

#### Comment on the effect of increasing the number of trees on validation performance. Compare the performance of the best performing GBDT against that of the best performing Random Forest classifier (Ex. 2a) and Decision Tree classifier trained with entropy (Ex. 1a).

**Answer:** [TODO]